### Setup — run this cell first
The uploaded exercise file refers to `initialize_parameters_deep`, `L_model_forward`,
`compare_with_autograd`, and several test tensors, but does not define them. The
following **self-contained helper implementations** supply these prerequisites so
**Run All** works without an earlier lecture notebook.

**Tensor convention used throughout:** every row is one training example.
Therefore `X` has shape `(m, n₀)`, `W[l]` has shape `(n_(l-1), n_l)`, and
`b[l]` has shape `(1, n_l)`. Hidden layers use ReLU in the baseline network;
the output layer uses sigmoid and the loss is **mean binary cross-entropy**.
These helpers are a compatible reconstruction, not copied from an absent course file.


In [1]:
import math
import torch

# Double precision makes the autograd comparison more reliable, especially
# when checking very small differences in the gradients.
DTYPE = torch.float64


def initialize_parameters_deep(layer_dims, seed=42):
    """Create W and b for each layer using the row-example convention."""
    generator = torch.Generator().manual_seed(seed)  # Reproducible results
    params = {}
    for l in range(1, len(layer_dims)):
        n_previous, n_current = layer_dims[l - 1], layer_dims[l]
        # He scaling keeps ReLU activations at a reasonable size.
        params[f"W{l}"] = (
            torch.randn(n_previous, n_current, generator=generator, dtype=DTYPE)
            * math.sqrt(2.0 / n_previous)
        )
        # A single bias row broadcasts across the entire batch.
        params[f"b{l}"] = torch.zeros(1, n_current, dtype=DTYPE)
    return params


def linear_activation_forward(A_prev, W, b, activation):
    """Apply a linear step followed by the chosen activation."""
    Z = A_prev @ W + b                 # Z = A_previous * W + b
    if activation == "relu":
        A = torch.relu(Z)
    elif activation == "leaky_relu":
        A = torch.where(Z > 0, Z, 0.01 * Z)
    elif activation == "sigmoid":
        A = torch.sigmoid(Z)
    else:
        raise ValueError(f"Unknown activation: {activation}")
    # Cache values needed for the backward pass; preserve the exercise format.
    return A, ((A_prev, W, b), Z)


def linear_activation_backward(dA, cache, activation):
    """Backpropagate through an activation and a linear layer."""
    (A_prev, W, b), Z = cache
    if activation == "relu":
        # ReLU derivative: 1 for positive Z, otherwise 0.
        dZ = dA * (Z > 0).to(Z.dtype)
    elif activation == "leaky_relu":
        # Leaky ReLU derivative: 1 if Z > 0, else 0.01.
        dZ = dA * torch.where(Z > 0, torch.ones_like(Z),
                              torch.full_like(Z, 0.01))
    elif activation == "sigmoid":
        S = torch.sigmoid(Z)
        dZ = dA * S * (1 - S)         # Chain rule: sigmoid derivative
    else:
        raise ValueError(f"Unknown activation: {activation}")

    m = A_prev.shape[0]             # Number of examples is number of ROWS
    dW = A_prev.T @ dZ / m          # Gradient for layer weights
    db = dZ.sum(dim=0, keepdim=True) / m  # Gradient for layer biases
    dA_prev = dZ @ W.T              # Pass gradient to previous layer
    return dA_prev, dW, db


def L_model_forward(X, params):
    """Baseline network: ReLU hidden layers and sigmoid output."""
    A = X
    caches = []
    L = len(params) // 2
    for l in range(1, L):
        A, cache = linear_activation_forward(
            A, params[f"W{l}"], params[f"b{l}"], "relu"
        )
        caches.append(cache)
    AL, cache = linear_activation_forward(
        A, params[f"W{L}"], params[f"b{L}"], "sigmoid"
    )
    caches.append(cache)
    return AL, caches


def L_model_backward(AL, Y, caches):
    """Correct baseline backward pass, used for interpreting Exercise 3."""
    grads = {}
    # Derivative of binary cross-entropy *before* averaging over examples.
    safe_AL = AL.clamp(1e-12, 1 - 1e-12)
    dA = -(Y / safe_AL - (1 - Y) / (1 - safe_AL))
    L = len(caches)
    for l in range(L, 0, -1):
        activation = "sigmoid" if l == L else "relu"
        dA, dW, db = linear_activation_backward(dA, caches[l - 1], activation)
        grads[f"dW{l}"] = dW
        grads[f"db{l}"] = db
    return grads


def compare_with_autograd(forward_fn, backward_fn, params, X, Y):
    """Compare handwritten gradients to PyTorch autograd for each parameter."""
    # Give autograd its own independent differentiable parameter tensors.
    checked_params = {
        name: value.detach().clone().requires_grad_(True)
        for name, value in params.items()
    }
    AL, caches = forward_fn(X, checked_params)
    safe_AL = AL.clamp(1e-12, 1 - 1e-12)
    loss = -(Y * safe_AL.log() + (1 - Y) * (1 - safe_AL).log()).mean()
    loss.backward()                 # PyTorch computes the reference gradients

    # Our backward function computes gradients from its cached forward values.
    manual_grads = backward_fn(AL.detach(), Y, caches)
    differences = {}
    for l in range(1, len(checked_params) // 2 + 1):
        for kind in ("W", "b"):
            parameter_name = f"{kind}{l}"
            gradient_name = f"d{parameter_name}"
            reference = checked_params[parameter_name].grad
            calculated = manual_grads[gradient_name]
            # Max absolute error: near 0 means our derivative is correct.
            error = (calculated - reference).abs().max().item()
            differences[gradient_name] = error
            print(f"{gradient_name:4s} shape {tuple(calculated.shape)!s:8s} "
                  f"| max difference from autograd {error:.2e}")
    return differences


# Small reproducible test network: input=4, hidden=5 and 3, output=1.
# The original file referenced these variables but did not provide them.
params_small = initialize_parameters_deep([4, 5, 3, 1], seed=7)
random_gen = torch.Generator().manual_seed(123)
X_small = torch.randn(10, 4, generator=random_gen, dtype=DTYPE)
Y_small = torch.randint(0, 2, (10, 1), generator=random_gen).to(DTYPE)

print("Setup ready: PyTorch", torch.__version__)
print("Small input:", tuple(X_small.shape), "| labels:", tuple(Y_small.shape))


Setup ready: PyTorch 2.14.1
Small input: (10, 4) | labels: (10, 1)


---
## ★ Exercises (on your own)

### Exercise 1 · Shapes and number of parameters
For `layer_dims = [784, 64, 32, 1]` and a batch of $m = 10000$ examples:

**(a)** For every layer, print the shapes of $\mathbf{W}^{[l]}$, $\mathbf{b}^{[l]}$ and $\mathbf{Z}^{[l]}$, **without** running the network: compute them from `layer_dims` and `m`.

**(b)** Print the total number of parameters (all the entries of all the $\mathbf{W}$ and $\mathbf{b}$), then check it by counting the entries of `initialize_parameters_deep(layer_dims)`.

**(c)** Do the same count for `[784, 20, 7, 5, 1]`. Which network has more parameters, and roughly how many times more?

Expected output:
```
layer 1: W (784, 64)  b (1, 64)  Z (10000, 64)
layer 2: W (64, 32)   b (1, 32)  Z (10000, 32)
layer 3: W (32, 1)    b (1, 1)   Z (10000, 1)
total parameters: 52353 | counted: 52353
[784, 20, 7, 5, 1]: 15893 | ratio 3.3
```

In [2]:
# Exercise 1: shapes and number of parameters (NO forward pass needed).
layer_dims = [784, 64, 32, 1]  # Input: 784 features, two hidden layers, 1 output
m = 10_000                     # Number of examples (rows)

# (a) Each weight matrix connects the previous layer to the current layer.
for l in range(1, len(layer_dims)):
    n_prev, n_curr = layer_dims[l - 1], layer_dims[l]
    W_shape = (n_prev, n_curr)  # We use X @ W: (m, n_prev) @ (n_prev, n_curr)
    b_shape = (1, n_curr)      # A single row of biases broadcasts to all m
    Z_shape = (m, n_curr)      # Every example produces n_curr outputs
    print(f"layer {l}: W {W_shape}  b {b_shape}  Z {Z_shape}")

# (b) A layer has n_prev*n_curr weight entries PLUS n_curr biases.
formula_total = sum(
    layer_dims[l - 1] * layer_dims[l] + layer_dims[l]
    for l in range(1, len(layer_dims))
)
params = initialize_parameters_deep(layer_dims)
counted_total = sum(tensor.numel() for tensor in params.values())
print(f"total parameters: {formula_total} | counted: {counted_total}")
assert formula_total == counted_total == 52_353

# (c) Repeat for a thinner/deeper network and compare parameter counts.
other_dims = [784, 20, 7, 5, 1]
other_formula = sum(
    other_dims[l - 1] * other_dims[l] + other_dims[l]
    for l in range(1, len(other_dims))
)
other_params = initialize_parameters_deep(other_dims)
other_counted = sum(tensor.numel() for tensor in other_params.values())
ratio = formula_total / other_formula  # Compare original to alternate network
print(f"{other_dims}: {other_formula} | ratio {ratio:.1f}")
assert other_formula == other_counted == 15_893

# Answer: [784, 64, 32, 1] has more parameters, about 3.3 times as many.


layer 1: W (784, 64)  b (1, 64)  Z (10000, 64)
layer 2: W (64, 32)  b (1, 32)  Z (10000, 32)
layer 3: W (32, 1)  b (1, 1)  Z (10000, 1)
total parameters: 52353 | counted: 52353
[784, 20, 7, 5, 1]: 15893 | ratio 3.3


### Exercise 2 · Leaky ReLU
Leaky ReLU is $g(z) = z$ if $z > 0$, else $0.01\,z$. Its derivative is 1 if $z > 0$, else $0.01$. Unlike ReLU, it never has a zero gradient.

**(a)** Write `leaky_relu(Z)` (returns `(A, Z)`) and `leaky_relu_backward(dA, cache)`.

**(b)** Write `L_model_forward_leaky(X, params)` and `L_model_backward_leaky(AL, Y, caches)`: copies of `L_model_forward` and `L_model_backward` with leaky ReLU in the hidden layers. *Hint:* the simplest way is to add a `"leaky_relu"` case to copies of `linear_activation_forward` and `linear_activation_backward`.

**(c)** Check your gradients with `compare_with_autograd(L_model_forward_leaky, L_model_backward_leaky, params_small, X_small, Y_small)`. Every difference should be about $10^{-7}$ or smaller; your tiny numbers may not match the expected output exactly.

Expected output of (c):
```
dW1  shape (4, 5)  | max difference from autograd 3.0e-08
db1  shape (1, 5)  | max difference from autograd 1.5e-08
dW2  shape (5, 3)  | max difference from autograd 1.5e-08
db2  shape (1, 3)  | max difference from autograd 1.5e-08
dW3  shape (3, 1)  | max difference from autograd 4.5e-08
db3  shape (1, 1)  | max difference from autograd 1.1e-08
```

In [3]:
# Exercise 2(a): Leaky ReLU activation and its derivative.
def leaky_relu(Z, slope=0.01):
    """Return Leaky ReLU output and Z (cache) needed for backpropagation."""
    # Positive Z stays unchanged; negative/zero Z is multiplied by 0.01.
    A = torch.where(Z > 0, Z, slope * Z)
    return A, Z


def leaky_relu_backward(dA, cache, slope=0.01):
    """Compute dZ = dA * g'(Z), where g is Leaky ReLU."""
    Z = cache                  # Forward step stored its original input
    # d/dZ (Leaky ReLU) = 1 for Z > 0; otherwise it equals `slope`.
    local_gradient = torch.where(
        Z > 0, torch.ones_like(Z), torch.full_like(Z, slope)
    )
    return dA * local_gradient  # Elementwise chain rule


# Quick check: negative inputs retain a small nonzero gradient.
values = torch.tensor([[-2.0, 0.0, 3.0]], dtype=DTYPE)
activations, leaky_cache = leaky_relu(values)
print("Leaky ReLU test output: ", activations.tolist())
print("Leaky ReLU test dZ:     ",
      leaky_relu_backward(torch.ones_like(values), leaky_cache).tolist())


# Exercise 2(b): Use Leaky ReLU for EVERY hidden layer and sigmoid for output.
def L_model_forward_leaky(X, params):
    """Forward pass for the full neural network with Leaky ReLU hidden layers."""
    A = X
    caches = []
    L = len(params) // 2
    for l in range(1, L):
        # The shared helper supports leaky_relu and returns the expected cache.
        A, cache = linear_activation_forward(
            A, params[f"W{l}"], params[f"b{l}"], "leaky_relu"
        )
        caches.append(cache)
    # Binary classification output: sigmoid, not Leaky ReLU.
    AL, cache = linear_activation_forward(
        A, params[f"W{L}"], params[f"b{L}"], "sigmoid"
    )
    caches.append(cache)
    return AL, caches


def L_model_backward_leaky(AL, Y, caches):
    """Backward pass with Leaky ReLU derivative at hidden layers."""
    grads = {}
    L = len(caches)
    safe_AL = AL.clamp(1e-12, 1 - 1e-12)
    # dLoss / dAL for binary cross-entropy; the helper later divides by m.
    dA = -(Y / safe_AL - (1 - Y) / (1 - safe_AL))
    for l in range(L, 0, -1):
        # Sigmoid only at the final layer; Leaky ReLU in all hidden layers.
        activation = "sigmoid" if l == L else "leaky_relu"
        dA, dW, db = linear_activation_backward(dA, caches[l - 1], activation)
        grads[f"dW{l}"] = dW
        grads[f"db{l}"] = db
    return grads


# Exercise 2(c): Cross-check our chain-rule gradients with automatic differentiation.
print("\nGradient check for Leaky ReLU network:")
leaky_differences = compare_with_autograd(
    L_model_forward_leaky, L_model_backward_leaky, params_small, X_small, Y_small
)
assert max(leaky_differences.values()) < 1e-7  # Should be essentially zero
print("All Leaky ReLU gradients passed the numerical check.")


Leaky ReLU test output:  [[-0.02, 0.0, 3.0]]
Leaky ReLU test dZ:      [[0.01, 0.01, 1.0]]

Gradient check for Leaky ReLU network:
dW1  shape (4, 5)   | max difference from autograd 2.78e-17
db1  shape (1, 5)   | max difference from autograd 1.39e-17
dW2  shape (5, 3)   | max difference from autograd 1.11e-16
db2  shape (1, 3)   | max difference from autograd 8.33e-17
dW3  shape (3, 1)   | max difference from autograd 1.04e-17
db3  shape (1, 1)   | max difference from autograd 2.78e-17
All Leaky ReLU gradients passed the numerical check.


### Exercise 3 · Find the bugs
The function below contains **two bugs**. Neither of them raises an error: the shapes are all correct, the gradients are just wrong.

**(a)** Run `compare_with_autograd(L_model_forward, L_model_backward_buggy, params_small, X_small, Y_small)`. Which gradients are wrong?

**(b)** Find the two bugs, write a fixed copy `L_model_backward_fixed`, and run the comparison again. All differences should be about $10^{-8}$.

**(c)** In a markdown cell, explain each bug in one sentence. One of them is a typical mistake when you copy code written for the **column** convention.

Expected output of (b):
```
dW1  shape (4, 5)  | max difference from autograd 1.5e-08
db1  shape (1, 5)  | max difference from autograd 7.5e-09
dW2  shape (5, 3)  | max difference from autograd 1.5e-08
db2  shape (1, 3)  | max difference from autograd 0.0e+00
dW3  shape (3, 1)  | max difference from autograd 7.5e-09
db3  shape (1, 1)  | max difference from autograd 3.7e-09
```


In [4]:
# Exercise 3(a): Keep the instructor's two bugs unchanged to expose them.
# Compare these WRONG gradients to the PyTorch autograd reference.
def L_model_backward_buggy(AL, Y, caches):
    grads = {}
    L = len(caches)
    m = AL.shape[1]
    dA = -(Y / AL - (1 - Y) / (1 - AL))
    for l in range(L, 0, -1):
        (A_prev, W, b), Z = caches[l - 1]
        if l == L:
            dZ = dA * torch.sigmoid(Z)
        else:
            dZ = dA * (Z > 0).float()
        grads[f"dW{l}"] = A_prev.T @ dZ / m
        grads[f"db{l}"] = dZ.sum(dim=0, keepdim=True) / m
        dA = dZ @ W.T
    return grads

compare_with_autograd(L_model_forward, L_model_backward_buggy, params_small, X_small, Y_small)
# Since both errors affect every backpropagated layer, all six gradients are wrong.


dW1  shape (4, 5)   | max difference from autograd 8.54e+00
db1  shape (1, 5)   | max difference from autograd 4.91e+00
dW2  shape (5, 3)   | max difference from autograd 2.45e+01
db2  shape (1, 3)   | max difference from autograd 9.43e+00
dW3  shape (3, 1)   | max difference from autograd 3.87e+00
db3  shape (1, 1)   | max difference from autograd 2.54e+00


{'dW1': 8.542842728369596,
 'db1': 4.910092660739694,
 'dW2': 24.47221703092559,
 'db2': 9.425092808575181,
 'dW3': 3.8727730267258313,
 'db3': 2.5442351686813414}

In [5]:
# Exercise 3(b): Fixed backward propagation, using the original cache structure.
def L_model_backward_fixed(AL, Y, caches):
    grads = {}
    L = len(caches)                      # Number of parameterized layers
    m = AL.shape[0]                      # FIX 1: examples are ROWS, not columns
    safe_AL = AL.clamp(1e-12, 1 - 1e-12)
    dA = -(Y / safe_AL - (1 - Y) / (1 - safe_AL))

    for l in range(L, 0, -1):           # Iterate backward from output layer
        (A_prev, W, b), Z = caches[l - 1]
        if l == L:
            S = torch.sigmoid(Z)
            # FIX 2: derivative of sigmoid is S*(1-S), not simply S.
            dZ = dA * S * (1 - S)
        else:
            # Hidden layers use ordinary ReLU (not Leaky ReLU here).
            dZ = dA * (Z > 0).to(Z.dtype)

        # Average gradients over the training examples.
        grads[f"dW{l}"] = A_prev.T @ dZ / m
        grads[f"db{l}"] = dZ.sum(dim=0, keepdim=True) / m
        # Transfer derivative to next layer in reverse order.
        dA = dZ @ W.T
    return grads


print("Gradient check AFTER fixing the bugs:")
fixed_differences = compare_with_autograd(
    L_model_forward, L_model_backward_fixed, params_small, X_small, Y_small
)
assert max(fixed_differences.values()) < 1e-7
print("All six gradients now agree with autograd.")


Gradient check AFTER fixing the bugs:
dW1  shape (4, 5)   | max difference from autograd 2.78e-17
db1  shape (1, 5)   | max difference from autograd 2.78e-17
dW2  shape (5, 3)   | max difference from autograd 5.55e-17
db2  shape (1, 3)   | max difference from autograd 1.73e-18
dW3  shape (3, 1)   | max difference from autograd 2.78e-17
db3  shape (1, 1)   | max difference from autograd 1.39e-17
All six gradients now agree with autograd.


### Exercise 3(c) — Explanation of the two bugs

1. **Wrong batch dimension:** `m = AL.shape[1]` counts output features instead of training examples; with examples in **rows**, the correct batch size is `m = AL.shape[0]`, otherwise every averaged gradient is scaled incorrectly (a common row-versus-column convention mistake).
2. **Wrong sigmoid derivative:** `dZ = dA * torch.sigmoid(Z)` is missing the derivative factor `(1 - torch.sigmoid(Z))`; the correct chain rule is `dZ = dA * S * (1 - S)` for `S = sigmoid(Z)`.

**Part (a):** With the buggy code, `dW1`, `db1`, `dW2`, `db2`, `dW3`, and `db3` all disagree with autograd, because these mistakes affect the output gradient and/or its propagation through the network. **Part (b):** After both fixes, all six differences are approximately zero (the exact printed values depend on the random test tensors and floating-point precision).
